In [ ]:
# 구글드라이브 연동 그리고 깃허브 클론/풀
import os
import sys
import shutil
from google.colab import drive

drive.mount('/content/drive')

%cd /content
if os.path.exists('/content/korean-chatbot'):
    %cd korean-chatbot
    !git pull
else:
    !git clone https://github.com/kkkk2058/korean-chatbot.git
    %cd korean-chatbot

!pip install -r requirements.txt

In [ ]:
import shutil, os

os.makedirs("models", exist_ok=True)
shutil.copy("/content/drive/MyDrive/korean-chatbot/models/vocab.json", "models/vocab.json")

os.makedirs("data", exist_ok=True)
shutil.copy("/content/drive/MyDrive/korean-chatbot/data/namuwiki.txt", "data/namuwiki.txt")

print("파일 로드 완료!")
path = "data/namuwiki.txt"
print(f"namuwiki.txt 크기: {os.path.getsize(path) / 1024 / 1024:.1f} MB")

In [ ]:
import torch
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"device: {device}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")

In [ ]:
import sys
sys.path.append('/content/korean-chatbot')
from src.tokenizer import BPETokenizer
from src.model import Transformer

tok = BPETokenizer()
tok.load("models/vocab.json")

model = Transformer(vocab_size=tok.tokenizer.get_vocab_size())
model = model.to(device)
print(f"vocab size: {tok.tokenizer.get_vocab_size()}")
print(f"파라미터 수: {sum(p.numel() for p in model.parameters()):,}")

In [ ]:
from torch.utils.data import Dataset, DataLoader
import torch

# 1. 데이터셋 클래스
class TextDataset(Dataset):
    def __init__(self, path, tokenizer, max_seq_len=512):
        self.samples = []
        
        # 1. 모든 줄 이어붙이기
        all_ids = []
        with open(path, "r", encoding="utf-8") as f:
            for line in f:
                ids = tokenizer.encode(line.strip())
                all_ids.extend(ids)
        
        print(f"전체 토큰 수: {len(all_ids):,}")
        
        # 2. 512씩 자르기
        for i in range(0, len(all_ids) - max_seq_len, max_seq_len):
            self.samples.append(torch.tensor(all_ids[i:i+max_seq_len]))
    
    def __len__(self):
        return len(self.samples)
    
    def __getitem__(self, idx):
        return self.samples[idx]

In [ ]:
# 2. 데이터로더
dataset = TextDataset("data/namuwiki.txt", tok)
dataloader = DataLoader(dataset, batch_size=16, shuffle=True)
print(f"총 샘플 수: {len(dataset):,}")

# 3. 옵티마이저
optimizer = torch.optim.AdamW(model.parameters(), lr=3e-4)


In [ ]:
EPOCHS = 3

for epoch in range(EPOCHS):
    total_loss = 0
    for step, batch in enumerate(dataloader):
        batch = batch.to(device)
        
        loss = model.loss(batch)
        
        optimizer.zero_grad()
        loss.backward()
        optimizer.step()
        
        total_loss += loss.item()
        
        if step % 100 == 0:
            print(f"epoch {epoch+1} | step {step} | loss {loss.item():.4f}")
    
    avg_loss = total_loss / len(dataloader)
    print(f"epoch {epoch+1} 완료 | 평균 loss: {avg_loss:.4f}")

print("학습 완료!")

In [ ]:
import shutil

os.makedirs("models", exist_ok=True)
torch.save(model.state_dict(), "models/model.pt")

# Drive 백업
os.makedirs("/content/drive/MyDrive/korean-chatbot/models", exist_ok=True)
shutil.copy("models/model.pt", "/content/drive/MyDrive/korean-chatbot/models/model.pt")
print("모델 저장 & Drive 백업 완료!")